In [1]:
# Estrategia Lay 0x1

import pandas as pd
import numpy as np

# Mostar todas as colunas
pd.set_option('display.max_columns', None)

In [7]:
data = pd.read_csv("../../data_total/dados_betfair.csv", sep=";")

In [14]:
# Filtar colunas para análise
datatest = data[['League', 'Home', 'Away', 'Goals_H_HT', 'Goals_A_HT', 'Goals_H_FT', 'Goals_A_FT', 'Goals_Min_H', 'Odd_H_Back', 'Odd_A_Back', 'Odd_CS_0x1_Lay']].copy()

#datatest.to_csv("TEBF002_Lay_0x1.csv", sep=";", index=False)

# Verificar se o placar FT foi 0x1
datatest['WCS'] = datatest.apply(lambda row: 0 if row['Goals_H_FT'] == 0 and row['Goals_A_FT'] == 1 else 1, axis=1)

# Definir o valor da aposta
STAKE = 1
COMISSAO = 0.065

# Criar o 'profit'
datatest['Profit'] = round(datatest.apply(lambda row: 
    -STAKE * (row['Odd_CS_0x1_Lay'] - 1) if row['WCS'] == 0  
    else STAKE * (1 - COMISSAO), 
    axis=1), 2)

# Adicionar coluna com o minuto do primeiro gol
datatest['Min_Goal_0x0'] = np.where(
    (datatest['Goals_H_HT'] == 0) & (datatest['Goals_A_HT'] == 0),
    datatest['Goals_Min_H'].apply(lambda x: x[1:3] if len(x) > 0 else 0),
    0
)

# Alterar [ para 0
datatest['Min_Goal_0x0'] = datatest['Min_Goal_0x0'].str.replace(']', '0')

# Alterar NaN para 0
datatest['Min_Goal_0x0'] = datatest['Min_Goal_0x0'].fillna(0)

# Transformar a coluna em inteiro
datatest['Min_Goal_0x0'] = datatest['Min_Goal_0x0'].astype(int)


datatest.head(15)

,League,Home,Away,Goals_H_HT,Goals_A_HT,Goals_H_FT,Goals_A_FT,Goals_Min_H,Odd_H_Back,Odd_A_Back,Odd_CS_0x1_Lay,WCS,Profit,Min_Goal_0x0
0,SPAIN 1,Ourense CF,Ponferradina,0,0,0.0,0.0,[],2.18,2.88,1000.0,1,0.94,0
1,SPAIN 1,Ponferradina,Zamora,1,0,2.0,0.0,"[36, 87]",2.28,3.50,11.0,1,0.94,0
2,SPAIN 1,Algeciras,Alcorcon,1,0,2.0,0.0,"[24, 90]",1.39,1.35,0.0,1,0.94,0
3,SPAIN 1,Zamora,Amorebieta,3,0,5.0,0.0,"[7, 13, 36, 54, 89]",1.94,3.10,1000.0,1,0.94,0
4,SPAIN 1,Gimnastic,Celta Vigo B,0,0,1.0,0.0,[54],2.26,2.46,1000.0,1,0.94,54
5,SPAIN 1,Ponferradina,Gimnastic,0,0,0.0,1.0,[],2.30,3.15,11.5,0,-10.50,0
6,SPAIN 1,Amorebieta,Celta Vigo B,3,1,4.0,3.0,"[9, 13, 34, 68]",1.01,1.02,1000.0,1,0.94,0
7,SPAIN 1,Gimnastic,Ponferradina,1,1,5.0,1.0,"[44, 50, 60, 75, 90]",2.42,3.45,12.5,1,0.94,0
8,SPAIN 1,Villarreal B,Recreativo Huelva,0,0,1.0,0.0,[58],2.00,3.40,980.0,1,0.94,58
9,SPAIN 1,Lugo,Barakaldo,0,0,0.0,0.0,[],1.65,1.24,1000.0,1,0.94,0


In [15]:
# Função para criar faixas de odds
def criar_faixa_h_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
    
def criar_faixa_a_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
        
def criar_faixa_cs_lay(odd):
    if odd < 9.0:
        return '8.0-8.9'
    elif odd < 10.0:
        return '9.0-9.9'
    elif odd < 11.0:
        return '10.0-10.9'
    elif odd < 12.0:
        return '11.0-11.9'
    elif odd < 13.0:
        return '12.0-12.9'
    elif odd < 14.0:
        return '13.0-13.9'
    elif odd < 15.0:
        return '14.0-14.9'
    elif odd < 16.0:
        return '15.0-15.9'
    elif odd < 18.0:
        return '16.0-17.9'
    elif odd < 20.0:
        return '18.0-19.9'
    else:
        return '20.0+'
    
# Aplicar as funções às colunas correspondentes
datatest['Faixa_Odd_H_Back'] = datatest['Odd_H_Back'].apply(criar_faixa_h_back)
datatest['Faixa_Odd_A_Back'] = datatest['Odd_A_Back'].apply(criar_faixa_a_back)
datatest['Faixa_Odd_CS_0x1_Lay'] = datatest['Odd_CS_0x1_Lay'].apply(criar_faixa_cs_lay)

# Agrupars por faixas e calcular estatísticas
print("\n📊 ANÁLISE POR FAIXAS (recomendado para amostras maiores):")
print("-" * 80)

agrupando_faixas = datatest.groupby(['Faixa_Odd_H_Back', 'Faixa_Odd_A_Back', 'Faixa_Odd_CS_0x1_Lay']).agg(
    Total_Jogos=('WCS', 'count'),
    Jogos_0x1=('WCS', 'sum'),
    Percentual_Acerto=('WCS', lambda x: (x.sum() / len(x) * 100)),
    Lucro_Total=('Profit', 'sum')
).reset_index()

# Arredondar valores
agrupando_faixas['Percentual_Acerto'] = agrupando_faixas['Percentual_Acerto'].round(2)

# Agrupar por lucro Total
agrupando_faixas = agrupando_faixas.sort_values(by='Lucro_Total', ascending=False)

agrupando_faixas.head(10)



📊 ANÁLISE POR FAIXAS (recomendado para amostras maiores):
--------------------------------------------------------------------------------


,Faixa_Odd_H_Back,Faixa_Odd_A_Back,Faixa_Odd_CS_0x1_Lay,Total_Jogos,Jogos_0x1,Percentual_Acerto,Lucro_Total
34,1.80-2.09,4.00-4.99,13.0-13.9,65,64,98.46,48.16
79,2.10-2.49,3.50-3.99,16.0-17.9,34,34,100.00,31.96
107,2.50-2.99,2.50-2.99,11.0-11.9,45,44,97.78,30.86
45,1.80-2.09,5.00+,15.0-15.9,44,43,97.73,26.42
200,5.00+,1.50-1.79,10.0-10.9,37,36,97.30,24.84
14,1.50-1.79,5.00+,16.0-17.9,96,92,95.83,23.48
78,2.10-2.49,3.50-3.99,15.0-15.9,24,24,100.00,22.56
44,1.80-2.09,5.00+,14.0-14.9,39,38,97.44,22.22
137,3.00-3.49,2.10-2.49,12.0-12.9,23,23,100.00,21.62
74,2.10-2.49,3.50-3.99,11.0-11.9,83,78,93.98,21.32


In [ ]:
# Selecionar apenas as 8 linhas com maior lucro total
top_10_lucro = agrupando_faixas.head(10)

# Total de jogos
total_jogos = top_10_lucro['Total_Jogos'].sum()
print(f"📈 Total de Jogos: {total_jogos}")

# Total de acertos
total_acertos = top_10_lucro['Jogos_0x1'].sum()
print(f"✅ Total de Acertos: {total_acertos}")

# Percentual de acerto
percentual_acerto = (total_acertos / total_jogos) * 100
print(f"🎯 Percentual de Acerto: {percentual_acerto:.2f}%")

# Total de Lucro
total_lucro = top_10_lucro['Lucro_Total'].sum()
print(f"💰 Total de Lucro: {total_lucro:.2f}")


📈 Total de Jogos: 490
✅ Total de Acertos: 476
🎯 Percentual de Acerto: 97.14%
💰 Total de Lucro: 273.44
📉 Total de Perda: -0.00


In [8]:
condicoes = [
    # Linha 1
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'] >= 20.0)),
    
    # Linha 2
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(13.0, 13.9))),
    
    # Linha 3
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(3.50, 3.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(12.0, 12.9))),
    
    # Linha 4
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(18.0, 19.9))),
    
    # Linha 5
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(3.50, 3.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(15.0, 15.9))),
    
    # Linha 6
    ((datatest['Odd_H_Back'].between(2.50, 2.99)) & 
     (datatest['Odd_A_Back'].between(2.50, 2.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(11.0, 11.9))),
    
    # Linha 7
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'] >= 5.00) & 
     (datatest['Odd_CS_0x1_Lay'].between(15.0, 15.9))),
    
    # Linha 8
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'] >= 5.00) & 
     (datatest['Odd_CS_0x1_Lay'].between(14.0, 14.9))),
    
    # Linha 9
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(11.0, 11.9))),
    
    # Linha 10
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(3.50, 3.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(16.0, 17.9)))
]

# Aplicar as condições: Bet = 1 se qualquer uma das condições for verdadeira
datatest['Bet'] = np.where(pd.concat(condicoes, axis=1).any(axis=1), 1, 0)

# Calcular o lucro para cada jogo com base nas condições
datatest['PL'] = np.where(
    (datatest['Bet'] == 1) & (datatest['WCS'] == 1),
    datatest['Profit'],
    np.where(
        (datatest['Bet'] == 1) & (datatest['WCS'] == 0),
        - datatest['Odd_CS_0x1_Lay'] + 1,
        0
    )
)

datatest['PL_ACC'] = datatest['PL'].cumsum()

# Jogos No HT diferente de 0x0
datatest['HT_Dif_0x0'] = np.where((datatest['Goals_H_HT'] == 0) & (datatest['Goals_A_HT'] == 0), 1, 0)

# Jogos No HT diferente de 0x1
datatest['HT_Dif_0x1'] = np.where((datatest['Goals_H_HT'] == 0) & (datatest['Goals_A_HT'] == 1), 1, 0)

# Jogos no HT sem ser 0x0 ou 0x1
datatest['HT_Dif_0x0_0x1'] = np.where((datatest['HT_Dif_0x0'] == 1) | (datatest['HT_Dif_0x1'] == 1), 0, 1)

# Jogos no FT 0x0
datatest['FT_Dif_0x0'] = np.where((datatest['Goals_H_FT'] == 0) & (datatest['Goals_A_FT'] == 0), 1, 0)

# Jogos no FT 0x1
datatest['FT_Dif_0x1'] = np.where((datatest['Goals_H_FT'] == 0) & (datatest['Goals_A_FT'] == 1), 1, 0)

# Jogos mo HT 0x0 e no FT 0x1
datatest['HT_0x0_FT_0x1'] = np.where((datatest['HT_Dif_0x0'] == 1) & (datatest['FT_Dif_0x1'] == 1), 1, 0)

# Jogos no HT 0x1 e diferente no FT
datatest['HT_0x1_FT_Dif_0x1'] = np.where((datatest['HT_Dif_0x1'] == 1) & (datatest['FT_Dif_0x1'] == 0), 1, 0)

    
datatest.head(15)

datatest.to_csv("TEBF002_Lay_0x1_ANALISE.csv", sep=";", index=False)
    

In [9]:

# pegar os jogos que tiveram gols depois dos 75m Min_Goal_0x0
df_min = datatest[(datatest['Bet'] == 1) & (datatest['HT_Dif_0x0_0x1'] == 0) & (datatest['Min_Goal_0x0'] >= 75)]

df_min.to_csv("TEBF002_Lay_0x1_MINUTO.csv", sep=";", index=False)

# Agora jogos que estavam 0x1 e tiveram gols depois dos 75m Min_Goal_0x1
df_min_0x1 = datatest[(datatest['Bet'] == 1) & (datatest['HT_Dif_0x0_0x1'] == 1)]

df_min_0x1.to_csv("TEBF002_Lay_0x1_MINUTO_0x1.csv", sep=";", index=False)

In [10]:
# Total de jogos
total_jogos = datatest['Bet'].sum()
print(f"📈 Total de Jogos: {total_jogos}")

# Total de acertos
total_acertos = datatest[datatest['Bet'] == 1]['WCS'].sum()
print(f"✅ Total de Acertos: {total_acertos}")

# Percentual de acerto
percentual_acerto = (total_acertos / total_jogos) * 100
print(f"🔍 Percentual de Acerto: {percentual_acerto:.2f}%")

# Total de Lucro
total_lucro = datatest['PL'].sum()
print(f"💰 Total de Lucro: {total_lucro:.2f}")

# Odd Média
odd_media = datatest[datatest['Bet'] == 1]['Odd_CS_0x1_Lay'].mean()
print(f"📊 Odd Média: {odd_media:.2f}")

# Drawndown
drawdown = datatest['PL_ACC'] - datatest['PL_ACC'].cummax()
max_drawdown = drawdown.min()  # ou .max() dependendo da convenção
print(f"📉 Max Drawdown: {max_drawdown:.2f}%")

# Jogos no HT 0x0, se Bet == 1 & HT_Dif_0x0 == 1
jogos_ht_0x0 = datatest[(datatest['Bet'] == 1) & (datatest['HT_Dif_0x0'] == 1)].shape[0]
print(f"📊 Jogos no HT 0x0, se Bet == 1 & HT_Dif_0x0 == 1: {jogos_ht_0x0}")

# Jogos no HT 0x1, se Bet == 1 & HT_Dif_0x1 == 1
jogos_ht_0x1 = datatest[(datatest['Bet'] == 1) & (datatest['HT_Dif_0x1'] == 1)].shape[0]
print(f"📊 Jogos no HT 0x1, se Bet == 1 & HT_Dif_0x1 == 1: {jogos_ht_0x1}")

# Jogos no HT sem ser 0x0 ou 0x1
jogos_ht_0x0_0x1 = datatest[(datatest['Bet'] == 1) & (datatest['HT_Dif_0x0_0x1'] == 1)].shape[0]
print(f"📊 Jogos no HT sem ser 0x0 ou 0x1, se Bet == 1 & HT_Dif_0x0_0x1 == 1: {jogos_ht_0x0_0x1}")

# Jogos no FT 0x0, se Bet == 1 & FT_Dif_0x0 == 1
jogos_ft_0x0 = datatest[(datatest['Bet'] == 1) & (datatest['FT_Dif_0x0'] == 1)].shape[0]
print(f"📊 Jogos no FT 0x0, se Bet == 1 & FT_Dif_0x0 == 1: {jogos_ft_0x0}")

# Jogos no HT 0x0 e No FT 0x1
jogos_ht_0x0_ft_0x1 = datatest[(datatest['Bet'] == 1) & (datatest['HT_0x0_FT_0x1'] == 1)].shape[0]
print(f"📊 Jogos no HT 0x0 e No FT 0x1, se Bet == 1 & HT_0x0_FT_0x1 == 1: {jogos_ht_0x0_ft_0x1}")

# Jogo no HT 0x1 eno FT 0x1
jogos_ht_0x1_ft_0x1 = datatest[(datatest['Bet'] == 1) & (datatest['FT_Dif_0x1'] == 1)].shape[0]
print(f"📊 Jogos no HT 0x1 e No FT 0x1, se Bet == 1 & HT_0x1_FT_Dif_0x1 == 1: {jogos_ht_0x1_ft_0x1}")

# Media do minuto gol 0x0 , se Bet == 1 e HT_Dif_0x1_0x0 == 0
media_minuto_gol_0x0 = datatest[(datatest['Bet'] == 1) & (datatest['HT_Dif_0x0_0x1'] == 0)]['Min_Goal_0x0'].mean()
print(f"📊 Media do minuto gol 0x0 , se Bet == 1 e HT_Dif_0x1_0x0 == 0: {media_minuto_gol_0x0:.2f}")


📈 Total de Jogos: 525
✅ Total de Acertos: 507
🔍 Percentual de Acerto: 96.57%
💰 Total de Lucro: 66.08
📊 Odd Média: 24.95
📉 Max Drawdown: -156.12%
📊 Jogos no HT 0x0, se Bet == 1 & HT_Dif_0x0 == 1: 164
📊 Jogos no HT 0x1, se Bet == 1 & HT_Dif_0x1 == 1: 70
📊 Jogos no HT sem ser 0x0 ou 0x1, se Bet == 1 & HT_Dif_0x0_0x1 == 1: 291
📊 Jogos no FT 0x0, se Bet == 1 & FT_Dif_0x0 == 1: 35
📊 Jogos no HT 0x0 e No FT 0x1, se Bet == 1 & HT_0x0_FT_0x1 == 1: 11
📊 Jogos no HT 0x1 e No FT 0x1, se Bet == 1 & HT_0x1_FT_Dif_0x1 == 1: 18
📊 Media do minuto gol 0x0 , se Bet == 1 e HT_Dif_0x1_0x0 == 0: 30.49


In [ ]:
# Se 0x0 HT Red Max 6% | Min 4% ***Acima de 6% Red Sair da Posição
# Se 0x0 Até no Min. 65 Red Max 7% ***Acima de 7% Red sair da Posição
# Se 0x0 Até no Min. 80 Neutro

# Se 0x1 Ht Red Max 15% | ***Acima de 15% Red Sair da Posição
# Se 0x1 Até no Min. 65 Red Max 25% | ***Acima de 25% Red Sair da Posição



-----------